# Module 7: Programmatic Extraction

While `info()` is designed for humans to read styled reports, **`get()`** is designed for computers. It is the fundamental tool for extracting coordinates, names, and any other biological data for use in your Python scripts.

In this module, we will learn how to turn molecular systems into raw data (numpy arrays and lists).

In [ ]:
import molsysmt as msm
from molsysmt import systems
import numpy as np

lysozyme = systems['T4 lysozyme L99A']['181l.bcif.gz']

### 1. Extracting Single Attributes
You can request any attribute available for the system's form. If you ask for one thing, `get()` returns it directly. This remains form-independent: MolSysMT can use a declared internal conversion route when the source form has no direct getter for that attribute.

In [ ]:
# Get the names of the first 5 atoms
names = msm.get(lysozyme, selection=[0,1,2,3,4], atom_name=True)
print(f"Atom names: {names}")

# Get the total number of atoms
n_atoms = msm.get(lysozyme, element='system', n_atoms=True)
print(f"Total atoms: {n_atoms}")

### 2. Extracting Coordinates (The 3D Tensor)
Coordinates are the most common data to extract. Remember the shape: `[n_structures, n_atoms, 3]`.

In [ ]:
coords = msm.get(lysozyme, selection='molecule_type=="protein"', coordinates=True)
print(f"Coordinates shape: {coords.shape}")
print(f"First atom position: {coords[0, 0]}")

```{admonition} Glossary: Coordinates Tensor
:class: info
The standard 3D array structure used in MolSysMT to store spatial data. Its shape is always `[n_structures, n_atoms, 3]`, ensuring consistency across different file formats.
```

### Deriving Periodic Box Properties
When a molecular form provides a periodic box matrix, `get()` can derive its lengths, angles, shape, and volume consistently. Lengths use nanometers, angles use radians, and volume uses cubic nanometers.

In [ ]:
box_lengths, box_angles, box_volume = msm.get(
    lysozyme, element='system',
    box_lengths=True, box_angles=True, box_volume=True,
)
print(f"Box lengths: {box_lengths}")
print(f"Box angles: {box_angles}")
print(f"Box volume: {box_volume}")

### 3. Multiple Attributes at Once
If you ask for multiple attributes, `get()` returns a **list** of results.

In [ ]:
ids, names, types = msm.get(lysozyme, selection=[10, 20, 30], 
                            atom_id=True, atom_name=True, atom_type=True)

for i, n, t in zip(ids, names, types):
    print(f"ID: {i} | Name: {n} | Type: {t}")

A multiple-attribute request may also combine atom data with system metadata. MolSysMT keeps the requested order and evaluates an incompatible attribute at another element level only when the attribute catalog defines a single unambiguous level. Thus, with `element='atom'`, the atom selection affects `coordinates` but not a system-level attribute such as `structure_id`. Native Structures and MolSys objects also expose stored temperature and energy series; `total_energy` is available when both potential and kinetic energy are present.

Treat indices as part of the scientific input contract. Atom and structure indices are 0-based, non-negative, and bounded by their corresponding counts. A malformed query or invalid index raises `molsysmt.ArgumentError` consistently, even when MDTraj, MDAnalysis, NumPy, or an on-disk backend performs the underlying operation. The contract applies across retrieval, conversion, extraction, removal, mutation, iteration, summaries, and visualization. This stable boundary makes pipelines easier to diagnose and keeps backend details out of user error handling.

--- 

### 🏆 Challenge 5: The Data Scientist

1. Load the **Industrial Enzyme** (H5MSM form).
2. Extract the coordinates of all **Nitrogen** atoms (`atom_name=="N"`).
3. Use `numpy.mean()` on the extracted coordinates to find their center of geometry.
4. Extract the `group_name` and `group_id` for those same Nitrogen atoms and print them as a neat table.

Now that you can extract data, you need to understand the physical units of that data. See you in **Module 8: Unit Safety**.

---

```{key-takeaway}
`msm.get()` is the workhorse for data extraction. It converts complex molecular objects into standard Python types (lists and numpy arrays) ready for any scientific library.
```